# Explore ML results on the PVC
`~/work` is your PVC, so it holds results written by other Pods and Jobs. Run each cell with **Shift+Enter**.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RESULTS = Path.home() / "work" / "results"
runs = pd.DataFrame([{"run": f.parent.name, **{k: v for k, v in json.loads(f.read_text()).items() if k != "sweep"}}
                     for f in sorted(RESULTS.glob("*/metrics.json"))])
runs

In [ ]:
for f in sorted(RESULTS.glob("*/sweep.csv")):
    df = pd.read_csv(f)
    plt.plot(df["n_estimators"], df["test_accuracy"], marker="o", label=f.parent.name)
plt.xlabel("n_estimators"); plt.ylabel("test accuracy"); plt.legend(); plt.show()

Load a saved model and classify a few digits. An `InconsistentVersionWarning` here means training and analysis used different scikit-learn versions, which is a good reason to pin versions.

In [ ]:
import joblib
from sklearn.datasets import load_digits

model = joblib.load(sorted(RESULTS.glob("*/best_model.joblib"))[-1])
digits = load_digits()
preds = model.predict(digits.data[-8:])
fig, axes = plt.subplots(1, 8, figsize=(10, 1.6))
for ax, img, p in zip(axes, digits.images[-8:], preds):
    ax.imshow(img, cmap="gray_r"); ax.set_title(f"pred {p}"); ax.axis("off")
plt.show()